<a href="https://colab.research.google.com/github/cindyrenxy/ac209b-project/blob/emilyc/ms3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 3 Notebook — MiDaS Project

**Canvas Project Number:** Canvas Project #77  
**Group Members:** Emily Chen, Cindy Ren, Alice Wang
**Course / Milestone:** Milestone 3

This notebook is organized to match the milestone requirements:
1. Problem statement refinement and introduction  
2. Comprehensive EDA review  
3. Baseline model selection and justification  
4. Results interpretation and analysis  
5. Final model pipeline setup

## 0. Brief recap of Milestone 2 progress

**Draft recap**

In Milestone 2, we established the foundation for our project by defining a clear problem setting and conducting exploratory analysis on the MiDaS-60 dataset. Our goal is to compare the performance and representation learning of Vision Transformers (ViTs) and Convolutional Neural Networks (CNNs) in a structured, non-natural image domain (Minecraft).

We first introduced the dataset and verified its structure, confirming a balanced class distribution (60 classes with equal samples) and a predefined *train/test* split. Through visual inspection and metadata analysis, we observed that while block classes are defined by consistent local texture and geometry, the dataset contains substantial variability in global context, including biome type and lighting conditions (day vs. night). Importantly, these contextual factors are unevenly distributed and often correlated with specific classes, suggesting a risk that models may rely on spurious background cues rather than intrinsic object features.

Our exploratory data analysis (EDA) further examined pixel-level statistics and metadata distributions. These analyses highlighted that global brightness and environmental context vary significantly across samples, while intra-class visual patterns remain stable. Based on these findings, we concluded that block classification, rather than biome classification—is a more appropriate task for evaluating model architectures, as it requires robustness to contextual variation and emphasizes fine-grained visual features.

Guided by these insights, we designed a preprocessing pipeline that includes normalization (to reduce lighting variation) and carefully controlled data augmentation (e.g., slight cropping, flipping, and mild color jitter). These transformations aim to improve model robustness without distorting the structured patterns inherent in Minecraft blocks.

Overall, Milestone 2 clarified the challenges posed by the dataset and informed our modeling strategy. In particular, it highlighted the importance of disentangling local texture features from global contextual signals, which motivates our comparison between CNNs (local feature bias) and ViTs (global attention mechanisms) in subsequent modeling steps.

## 1. Problem statement refinement and introduction

### Project introduction

This project focuses on image-based classification using the MiDaS-60 dataset, which consists of Minecraft images labeled by block type. Our objective is to evaluate how well computer vision models can distinguish between visually similar categories in a structured, non-natural image domain. In particular, we aim to understand how different modeling choices interact with the dataset’s visual characteristics, including texture consistency and contextual variability.

### Refined research question

**Primary question:** Can a Vision Transformer outperform a Convolutional Neural Network for Minecraft block classification, and do interpretability methods (Attention Rollout vs. Grad-CAM) reveal meaningful differences in the visual representations learned by the two architectures in a non-natural image domain?

### Why this question matters

A well-defined problem is critical because the dataset contains significant contextual variation, such as biome differences and lighting conditions, that are not directly tied to the target labels. Our EDA shows that block-level labels correspond more closely to stable local visual patterns (e.g., texture and structure), whereas higher-level labels (such as biome) are less consistently reflected in the image content. Framing the problem at the block level reduces reliance on spurious background cues, improves interpretability, and leads to more meaningful evaluation of model performance and generalization


In [ ]:
# ==== Setup ====
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from collections import Counter

import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2A. Colab + GitHub setup

Run the cells below in **Google Colab** after you upload this notebook or open it from GitHub.  
They are written so you can clone your repo, switch to **your own branch**, and push `ms3.ipynb`.

> Replace the placeholders for:
> - `YOUR_REPO_URL`
> - `YOUR_BRANCH_NAME`
> - optional Git identity

In [ ]:
# ==== mount Drive  ====
from google.colab import drive
drive.mount('/content/drive')

## 3. Data description

**Dataset:** MiDaS-60  
**Source:** `https://github.com/MinecraftDataset/MiDaS/tree/main.`

### Summary
- The dataset contains labeled images organized into training and test folders.
- Each class corresponds to a visual category in the environment.
- Our current framing emphasizes **block classification**.
- The images appear to vary in texture, color, viewpoint, and scene complexity.

### Why this dataset is useful
This dataset allows us to test whether image features are sufficient for distinguishing meaningful environmental categories and whether label granularity affects modeling success.

### Notes / caveats
- Some classes may be visually similar.
- Class imbalance may affect training and evaluation.
- A biome-level formulation may introduce label noise if multiple visually distinct blocks are merged together.

In [ ]:
# ==== Define data paths ====
DATA_ROOT = '/content/drive/MyDrive/MiDaS'
SMALL_DIR = os.path.join(DATA_ROOT, 'MiDaS-60_small')
LARGE_DIR = os.path.join(DATA_ROOT, 'MiDaS-60_large')

# Choose one
DATA_DIR = SMALL_DIR  # or LARGE_DIR

TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'test')

print("Train dir exists:", os.path.exists(TRAIN_DIR), TRAIN_DIR)
print("Test dir exists:", os.path.exists(TEST_DIR), TEST_DIR)

In [ ]:
# ==== Load dataset ====
basic_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=basic_transform)
test_dataset  = datasets.ImageFolder(TEST_DIR, transform=basic_transform)

print(f"Loaded {len(train_dataset)} train images")
print(f"Loaded {len(test_dataset)} test images")
print(f"Number of classes: {len(train_dataset.classes)}")
print("Classes:", train_dataset.classes[:20], "..." if len(train_dataset.classes) > 20 else "")

In [ ]:
# ==== Basic dataset table ====
train_counts = Counter([label for _, label in train_dataset.samples])
test_counts = Counter([label for _, label in test_dataset.samples])

class_df = pd.DataFrame({
    "class_name": train_dataset.classes,
    "train_count": [train_counts.get(i, 0) for i in range(len(train_dataset.classes))],
    "test_count":  [test_counts.get(i, 0) for i in range(len(train_dataset.classes))],
})
class_df["total_count"] = class_df["train_count"] + class_df["test_count"]
class_df = class_df.sort_values("total_count", ascending=False).reset_index(drop=True)

display(class_df.head(20))
print("Total images:", class_df["total_count"].sum())

## 4. Comprehensive EDA review

In this section, summarize the most important EDA findings and explain how they influenced preprocessing and model choice.

### Points to discuss
- Which classes are most represented / underrepresented?
- Are some classes visually similar to each other?
- Does block-level classification appear more reasonable than biome-level classification?
- Do image dimensions or quality vary?
- What preprocessing decisions were motivated by these observations?

## 5. Feature engineering and preprocessing

### Current preprocessing decisions


### Why these choices make sense


## 6. Baseline model selection and justification

### Baseline choice
A simple but relevant baseline for image classification is a small CNN or a transfer-learning model such as ResNet18. This type of model is more appropriate than linear or logistic regression alone because the task depends heavily on spatial image structure.

### Justification
- **Simple enough** to train and explain
- **Relevant** for image data
- **Interpretable enough** at the level of confusion matrices and class-wise performance
- Serves as a practical benchmark before trying more advanced architectures

## 7. Results interpretation and analysis


## 8. Future directions

## 9. Final model pipeline setup
